# SAIL analysis - canonical scoring trace

Every number quoted in the manuscript comes from `sail_scored.json`,
which this notebook produces, and from nowhere else.
Same discipline as the GD+CITL analysis notebook: the results tree is immutable,
nothing here writes to it, the logic lives in versioned modules
(`code/analysis/score_revision.py`), and this notebook is a thin
trace over them - the sequence lives here, the code lives there.

**Structure.** Sections 0 to 5 build and verify the canonical dataset;
sections 6 and 7 build the manuscript from it.

| § | What it does | Cost |
|---|---|---|
| 0 | Configuration - every path, set once | instant |
| 1 | What gets scored, and under what rules | reading |
| 2 | Dry run: inventory every input, score nothing | seconds |
| 3 | Full scoring pass → `sail_scored.json` | tens of minutes |
| 4 | Quick-look sanity table | seconds |
| 5 | Cross-check against the GD+CITL pipeline | instant |
| 6 | Figures and tables, in manuscript order | minutes |
| 7 | Build manifest: what exists, what is outstanding | instant |

Sections 0 to 5 need re-running only when the results tree changes. Section 6
can be re-run freely: it reads the canonical dataset and never re-scores.

## 0 - Configuration

Every path set once, exported through `SAILREV_*` environment variables BEFORE
`score_revision` is imported, exactly as the GD+CITL notebook sets `GDCITL_*`
before importing `gdcitl`. No later cell hardcodes a path, and the deposit
re-points everything by editing this cell alone.

In [ ]:
import os, sys, json
from pathlib import Path

# ------------------------------
# 0 | Deposit paths (nothing to edit; SAIL_ROOT overrides)
# ------------------------------
ROOT = Path(os.environ.get('SAIL_ROOT', Path.cwd().resolve().parents[1]))
for p in (ROOT / 'code' / 'analysis', ROOT / 'code' / 'method'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import paths

RESULTS = paths.RESULTS
OUT = paths.OUTPUT
SCORED = paths.SCORED

# score_revision reads these at import, so no module carries a hardcoded path.
os.environ['SAILREV_RESULTS'] = str(RESULTS)
os.environ['SAILREV_REPO'] = str(ROOT)
os.environ['SAILREV_STOCK'] = str(paths.TARGETS)
OUT.mkdir(parents=True, exist_ok=True)

# Verification mode. When the deposited scored record already exists, this
# run regenerates everything into output/regenerated/ and leaves the deposit
# untouched, so the final cell can compare the two. A fresh tree scores in
# place, which is the original behaviour.
VERIFY_MODE = SCORED.exists()
OUT_LIVE = (OUT / 'regenerated') if VERIFY_MODE else OUT
OUT_LIVE.mkdir(parents=True, exist_ok=True)
os.environ['SAILREV_OUT'] = str(OUT_LIVE / 'sail_scored.json')
os.environ['SAILREV_ANALYSIS'] = str(OUT_LIVE)
if VERIFY_MODE:
    print('VERIFICATION MODE: deposited record found, regenerating into', OUT_LIVE)

_ip = get_ipython()
if 'autoreload' not in _ip.extension_manager.loaded:
    _ip.run_line_magic('load_ext', 'autoreload')
_ip.run_line_magic('autoreload', '2')

paths.report()


## 1 - What gets scored, and by what

One record per **(domain, physics, method, target)**, every metric from
`evaluate_methods.compute_metrics` - imported, never reimplemented.

| domain | physics | methods |
|---|---|---|
| simulation | ideal + faithful | gs, gd, gs_intensity, gd_intensity, transformer_per_target, transformer_batched |
| simulation | ideal only (by decision) | fno_scratch, fno_regress |
| bench | ideal + faithful | gs, gd, transformer_per_target, transformer_batched (replay), sail, batched_sail_750, batched_sail_2000, gs_citl_{random,warm}, gd_citl_{random,warm} |
| bench_replay | ideal + faithful | the 2026-08-04 single-alignment re-capture: gs_750, gs_10000, gd_750, gd_10000, transformer_per_target, transformer_batched, sail, sail_plus, batched_sail_750, batched_sail_2000, gs_citl_{random,warm}, gd_citl_{random,warm} - methods discovered from the capture files, not listed in code |

**Rules enforced in `score_revision.py`, stated here so a reader of this
notebook knows what the numbers mean:**

- **Resolution.** Target and reconstruction always share one grid:
  `(H·pad_factor, W·pad_factor)`, with `pad_factor` passed into
  `compute_metrics` so the efficiency support threshold stays
  resolution-invariant. Ideal and faithful are separate scoring problems -
  compare within a physics block, never across.
- **Captures.** Wherever a raw DSLR frame exists it is re-processed here
  through the one shared rig calibration and DC exclusion
  (centre (495,510), radius 20, both × pad_factor), so no asymmetry between
  methods can enter at scoring time. Records carry `capture_kind` so any
  processed-PNG fallback is visible, not silent.
- **Batched-sim transformer** is re-propagated from `{target}_best_phase.npy`
  through `physics.py` with the run's own forward-model settings - generation
  and scoring on the same model, per the `reconstruct_from_phase_np` contract.
- **GS+CITL best iteration** is not in the run logs (only GD's is); it is
  derived as the argmin of `gs_camera_feedback_convergence.csv`. This is also
  what confirms or corrects the "GS+CITL best iteration = 0" claim.
- **The revision's bench dataset IS bench_replay** (decision 2026-08-04; the
  §5b agreement gate passed: same stored holograms 11 days apart, median
  |delta| 0.14 dB, identical-array transformer pairs 0.03 dB). `sailrev`
  resolves domain `"bench"` to `bench_replay` and renames its gs_750/gd_750 to
  gs/gd, so every figure and table below reads the single-alignment re-capture
  with no per-figure changes. July-era captures remain readable as domain
  `"bench_july"`; nothing in the manuscript quotes them. Verified: the
  per-target transformer phase is bit-identical (sha256) between the July and
  10k sweeps, so the agreement pairs isolate the rig.
- **Adaptation decay, found by the agreement gate and worth reporting:**
  rig-agnostic phases (transformers, GS/GD) reproduce to about +/-0.03 dB
  across 11 days; camera-adapted solutions decay slightly (GD+CITL about
  -0.2 dB, SAIL family about -0.4 dB), in proportion to how aggressively they
  adapted. Quoted SAIL numbers are therefore two-weeks-stale and conservative.
  A same-day dust-clean re-capture moved every mean by at most 0.06 dB, so the
  decay is drift, not dust.
- **FNO is ideal-only** by decision (2026-08-03): its from-scratch quality is
  too far below every other method for a faithful re-run to inform anything.
- **Wall-clock** is collected into a separate `timings` table from the run
  logs (per-target SAIL, batched SAIL, GD+CITL) - the batched-SAIL
  amortisation argument is quoted from there, not from memory.

## 2 - Dry run

**Run this first, every time the results tree has changed.** It walks every
source tree, assembles the full work list, and prints found/missing per
(domain, physics, method) - so a moved folder, a partial run, or a stale glob
shows up as an explicit MISSING line here rather than as a silently thinner
dataset after twenty minutes of scoring. It reads directory listings and small
logs only; it loads no images and scores nothing.

In [ ]:
import score_revision

work = score_revision.run(dry_run=True)

## 3 - Full scoring pass

Expectations before running:

- The dry run above should show **18/18 inputs** for every row you expect to
  exist, and the missing list empty for those. Anything short of 18 will score
  as-is but must be resolved (or explained) before the row is quotable.
- Upper bound is ~648 records if every family exists under both physics;
  the dry run's planned count is the real expectation.
- Runtime is dominated by capture loading and the batched-sim re-propagation;
  expect tens of minutes on the laptop, not hours. Faithful records process
  2000×2000 arrays and are the slow half.
- Output goes ONLY to `analysis/sail_scored.json` (path from Cell 0).
  The results tree itself is never written to. Re-running overwrites the JSON
  atomically with a fresh `meta.written` stamp.
- `score_revision.run(only=("bench_sail", "sim_gd"))` restricts to named
  families while debugging; a restricted JSON is NOT the canonical dataset -
  re-run unrestricted before quoting anything.

In [ ]:
records = score_revision.run(dry_run=False)

## 4 - Quick-look sanity table

Eyeball check only, not a result: per (domain, physics, method) mean ± SD PSNR
across targets, plus record counts and the timings summary. Anything surprising
here gets investigated at the record level before any figure is built.

In [ ]:
import numpy as np, collections

d = json.loads(SCORED.read_text())
recs, timings = d["records"], d["timings"]
print(f"{len(recs)} records, meta.written = {d['meta']['written']}\n")

groups = collections.defaultdict(list)
for r in recs:
    groups[(r["domain"], r["physics"], r["method"])].append(r)

print(f"{'domain':10s} {'physics':8s} {'method':24s} {'n':>3s} "
      f"{'PSNR mean±SD':>16s} {'SSIM mean':>9s}")
for k in sorted(groups):
    g = groups[k]
    ps = np.array([r["psnr"] for r in g]); ss = np.array([r["ssim"] for r in g])
    flag = "" if len(g) == 18 else "   <- expected 18"
    print(f"{k[0]:10s} {k[1]:8s} {k[2]:24s} {len(g):3d} "
          f"{ps.mean():8.2f} ± {ps.std():5.2f} {ss.mean():9.3f}{flag}")

print("\ntimings (seconds, None = not parsed from log):")
tg = collections.defaultdict(list)
for t in timings:
    tg[(t["method"], t["physics"])].append(t["seconds"])
for k in sorted(tg):
    vals = [v for v in tg[k] if v is not None]
    miss = sum(1 for v in tg[k] if v is None)
    tot = f"total {sum(vals)/3600:.2f} hr" if vals else "no values"
    print(f"  {k[0]:20s} {k[1]:8s} n={len(tg[k]):2d} {tot}"
          + (f"  ({miss} unparsed)" if miss else ""))

## 5 - Cross-check against the GD+CITL pipeline

Independent verification that the scorer is correct. The related
publication's pipeline scored the same bench captures (plain GD and both
camera-in-the-loop arms), and its scored record is deposited beside this
tree as `results/GD+CITL/analysis/gdcitl_scored.json`. Every overlapping
bench record must agree to the bit, since both pipelines read the same
files. The reference's simulation family scores that paper's own
simulation runs, not this paper's matched-compute sweep, so it is out of
scope here and the cell says how many records that excludes.


In [ ]:
GDCITL_SCORED = RESULTS / "GD+CITL" / "analysis" / "gdcitl_scored.json"
TOL = 1e-6

# Bench families only. The reference file also carries a "simulation"
# family, scored from the GD+CITL paper's own simulation runs. This
# paper's canonical simulation domain is the matched-compute 10k sweep,
# a different set of runs, so those records have no counterpart here and
# that family is out of scope for this check. The bench families below
# are the SAME captures scored by both pipelines, so they must agree to
# the bit.
MAP = {"bench_plain_gd": ("bench", "gd"),
       "bench_citl_random_init": ("bench", "gd_citl_random"),
       "bench_citl_warm_start": ("bench", "gd_citl_warm")}
METRICS = ("psnr", "ssim", "nmse", "mse", "diffraction_efficiency")

if not GDCITL_SCORED.exists():
    print("Cross-check SKIPPED. gdcitl_scored.json is not present. It is the")
    print("scored record of the related publication's pipeline, deposited")
    print("here for this check and also published in its own deposit")
    print("(DOI 10.17863/CAM.132918).")
else:
    ref = [r for r in json.loads(GDCITL_SCORED.read_text())
           if r["source"] in MAP]
    skipped = len(json.loads(GDCITL_SCORED.read_text())) - len(ref)
    print(f"{skipped} reference record(s) outside the bench families "
          f"(the reference's simulation family), out of scope here.")
    new = {(r["domain"], r["physics"], r["method"], r["target"]): r
           for r in json.loads(SCORED.read_text())["records"]}

    matched = missing = comparisons = exact = 0
    worst = (-1.0, None)
    diffs = []
    for r in ref:
        dom, meth = MAP[r["source"]]
        key = (dom, r["physics"], meth, r["target"])
        n = new.get(key)
        if n is None:
            missing += 1
            print(f"  MISSING in new pass: {key}")
            continue
        matched += 1
        for m in METRICS:
            if m not in r or m not in n:
                continue
            comparisons += 1
            d = abs(float(r[m]) - float(n[m]))
            if d == 0.0:
                exact += 1
            if d > worst[0]:
                worst = (d, (key, m, float(r[m]), float(n[m])))
            if d > TOL:
                diffs.append((key, m, float(r[m]), float(n[m]), d))

    assert matched > 0, ("nothing overlapped -- the comparison is vacuous; "
                         "check that scoring produced the bench families")
    print(f"\n{matched} overlapping records compared, {missing} missing")
    expected = matched * len(METRICS)
    print(f"metric comparisons actually made: {comparisons} (expected {expected})")
    print(f"bit-identical: {exact}/{comparisons}")
    assert comparisons == expected, (
        "some metrics were skipped -- key names differ between the two "
        "scored files; this check is NOT valid until that is resolved")

    if worst[1]:
        (key, m, a, b) = worst[1]
        print(f"largest deviation: {worst[0]:.3e}  {key} {m}  "
              f"ref {a:.6f} vs new {b:.6f}")
    if diffs:
        print(f"\n** {len(diffs)} metric(s) beyond tol {TOL} -- "
              f"DO NOT proceed: **")
        for key, m, a, b, d in diffs[:15]:
            print(f"  {key} {m:24s} ref {a:.6f} new {b:.6f} (d={d:.2e})")
    else:
        print(f"ALL metrics agree within {TOL} -- scorer verified against "
              f"the GD+CITL pipeline on the shared bench records.")


## 5b - The single-alignment re-capture: agreement gate, then the converged baselines

**STATUS: the gate PASSED on 2026-08-04** (clean re-capture, newest
`replay_converged` run): rig reproducibility median |delta| 0.14 dB, 95th pct
0.75, identical-array transformer pairs +/-0.03. The switch is live in
`sailrev` (domain "bench" resolves to `bench_replay`), so everything below §6
already reads the new dataset. This section stays as the permanent record of
the gate and reprints both reports on every run.

The 2026-08-04 replay enters the dataset as domain `bench_replay`, scored by
the same §3 full pass. Two reports, in a deliberate order.

**The agreement gate comes first.** Where the same stored phase was
photographed in July and again on 2026-08-04 (transformer, SAIL, batched SAIL,
all four CITL arms), the PSNR delta measures rig state across eleven days and
nothing else. `gs_750`/`gd_750` are excluded from the headline: they replay
the 10k sweep's own random initialisations, so their deltas fold reseeding
into the number. The saturation-caveat cell (`transformer_batched`, faithful)
is excluded too. The result doubles as a finding: rig-agnostic phases
reproduce to +/-0.03 dB while camera-adapted solutions decay in proportion to
how aggressively they adapted (GD+CITL about -0.2, SAIL family about -0.4),
so the quoted SAIL numbers are stale-by-two-weeks and conservative.

**Then the full bench table: all 14 captured methods per physics**, every row
from `bench_replay` so every number shares one alignment (GS and GD at 750 and
10,000, both transformers, SAIL, SAIL+, both batched SAIL budgets, all four
CITL arms; `sail_plus` is tabled because it was captured, and stays out of the
manuscript regardless). The headline inside it: GD gains +17.75 dB from 750 to
10,000 iterations in ideal simulation (E5), and the `gd_10000` minus `gd_750`
delta on the bench says how much of that survives the optics: +0.04 dB.

In [ ]:
import replay_analysis

agreement   = replay_analysis.report_agreement()
print("=" * 74)
convergence = replay_analysis.report_convergence()

---
# 6 - Figures and tables, in manuscript order

From here down the notebook is the manuscript. Every artifact appears in the
order it appears in the paper, each with one markdown cell saying what it
shows and one code cell that builds it.

Three rules hold for everything below:

1. **Quantitative artifacts read `sail_scored.json` and nothing else.** No
   cell re-scores, and no cell reads a stored metric from an experiment folder.
   If a number is not in the canonical dataset it does not go in the paper.
2. **The logic lives in modules** under `code/analysis/`, one per
   artifact, versioned with the repo. This notebook is the sequence, not the
   implementation, so a figure can be rebuilt outside the notebook and the
   deposit can reproduce it without a kernel.
3. **Colour is decided once**, in `sailrev.METHOD_COLORS`, keyed by method.
   Violet is a method that is not ours, pink is one of ours, and darker within
   a hue means more machinery. A figure never picks its own colours.

Artifacts not yet implemented are listed in place and call `pending()`, which
prints what is missing and why rather than raising, so this notebook always
runs top to bottom and always tells you what is outstanding.

In [ ]:
import sailrev as S

FIGS   = OUT_LIVE / "figures"
TABLES = OUT_LIVE / "tables"
for d in (FIGS, TABLES):
    d.mkdir(parents=True, exist_ok=True)

BUILT, PENDING = [], []

def built(name):
    BUILT.append(name); print(f"  built: {name}")

def pending(name, reason, blocker=""):
    PENDING.append((name, reason, blocker))
    print(f"  PENDING {name}: {reason}" + (f"  [blocked on: {blocker}]" if blocker else ""))

S.check()

## 6.1 - Tables

**T1, full results.** Mean, SD, median and IQR for every method, both forward
models, simulation and bench, so both statistics are present everywhere. The
narrative quotes the median because n = 18 and the bench distributions are
skewed; the table lets a reader confirm the mean agrees in direction.

**T2, compute and cost.** The amortisation table, and the one most exposed to
a sceptical reader. It reports the campaign cost honestly: batched training is
*not* cheaper than per-target training, because both spend one camera exposure
per target per epoch and the camera dominates. What batched training buys is a
single model, so the column that carries the argument is the last one, the
marginal cost of one further target.

In [ ]:
import tab_results

tab_results.build(TABLES)
built("T1 results (simulation, bench)"); built("T2 compute and cost")

**T3, implementation settings**: every setting read from the
runs' own `run_configuration.json` snapshots rather than retyped, with an
all-runs-agree check per method family (a drifted run renders as `MIXED(...)`
and cannot survive proofreading). Five products: **T3a** learned-model
families, **T3b** classical GS/GD baselines, **T3c** shared hardware and the
two forward-model conditions, **T3d** the FNO baseline (standard unmodified
FNO only), **T3e** the training-regime map that resolves every regime name
to what supervises it and how many targets share one model. The build also cross-checks
`tab_results.CONSTANTS` against the configs, closing that open item on every
rebuild.

**T4, prior-art regime table**: every cited comparator with its
propagation regime and its bearing on the claim. This is the table that
makes the near-field argument checkable instead of assertable.


In [ ]:
import tab_settings, tab_regime

tab_settings.build(TABLES);  built("T3 implementation settings (a-e), config-sourced")
tab_regime.build(TABLES);    built("T4 prior-art propagation-regime table")


## 6.2 - Main figures

**Fig 1 | Architecture and pipeline.** Target to patch tokens to encoder to complex field to
phase-only hologram to Fraunhofer replay, with the camera-feedback loop and
the loss drawn explicitly. Illustration, not data: no dependency on the
scored dataset.

In [ ]:
pending("Fig 1 architecture and pipeline",
        "vector illustration assembled outside the notebook (Ext Fig 6 + workflow)",
        "drawn asset")

**Fig 2 | Simulation benchmark.** Transformer against GS and GD under both
forward models, with convergence and wall-clock.

Report both forward models. The faithful model compresses every method
(transformer 14.98 dB median against 52.13 ideal), so the ideal block alone
would misstate the finding.

In [ ]:
import fig_simulation

# Fig 2 and E5 are a pair: this is the operating point; E5 is what unbounded
# compute does to it.
fig_simulation.build(FIGS); built("Fig 2 simulation benchmark")

**Fig 3 | SAIL against every alternative.** Paired per-target differences between SAIL and all eight other bench methods,
under both forward models, with medians, seeded bootstrap intervals and paired
Wilcoxon tests.

The two GD+CITL rows hold camera adaptation constant and vary only the
architecture. `adaptation_contrasts()` additionally prints what adaptation
alone buys a classical method, which the figure does not show.

In [ ]:
import fig_disentangle

fig_disentangle.build(FIGS)
built("Fig 3 SAIL against every alternative")

**Fig 4 | One model, eighteen targets.** The generalisation lead, and the
claim no iterative method can match structurally. Batched SAIL against
per-target SAIL at both epoch budgets, the marginal cost of a new target, and
attention selectivity rising with task complexity.

The wording constraint from T2 applies here too: the claim is one shared
model and a marginal cost in milliseconds, never a training-time speedup.

In [ ]:
import fig_batched

fig_batched.build(FIGS)
built("Fig 4 one model, eighteen targets")

**Fig 5 | Wavefront correction under misalignment, SAIL alone.** Re-captured
2026-08-05 with the phase corrector removed. The Fourier lens was moved toward
the camera in one direction (0.05, 0.1, 1, 2 mm, caliper-set), and at every
position three holograms were recorded: the simulation-only hologram, the SAIL
hologram adapted to the aligned bench, and SAIL retrained at that position.
Two checks certify the day: the lens returned to zero and both stored holograms
were re-photographed, and the 2 mm-trained hologram was shown back at zero
(it should be worse there, which is what rules out "retrained holograms are
simply better holograms"). Scoring uses the same two functions as the canonical
dataset, on one frozen calibration.


In [ ]:
import aberration_analysis, fig_aberration

# The 2026-08-05 lens-defocus session: scores every capture with the SAME
# functions as the canonical dataset, prints the curve, the two return-to-zero
# checks and the converse control, then renders Fig 5 at the largest
# displacement through E8's ROI machinery. SAIL alone: no phase corrector.
aberration_analysis.score()
aberration_analysis.report()
fig_aberration.build_fig5(FIGS);  built("Fig 5 misalignment, SAIL alone")


## 6.3 - Extended Data

Completeness lives here so the main text stays legible.

- E1 FNO comparison
- E2 Patch-size sweep
- E3 Attention maps and bias-dominance control
- E4 Amplitude versus intensity targets
- E5 Convergence and compute, matched budget
- E6 Generalisation to unseen patterns
- E7 Optical setup
- E8 Qualitative grid, all 18 targets, PSNR annotated
- E9 Aberration severity sweep

**E1 note.** At modes = 500 the regression arm reproduces GD's phase (35.32
against 35.64 dB), so the failure is trainability from scratch (10.48 dB),
not representational capacity.

In [ ]:
import fig_fno, fig_patch, fig_target_formulation, fig_convergence
import fig_generalisation
import fig_attention, fig_grid

fig_fno.build(FIGS);                built("E1 FNO")
fig_patch.build(FIGS);              built("E2 patch sweep")
fig_target_formulation.build(FIGS); built("E4 amplitude vs intensity")

# E5 reads the sweep aggregates, not sail_scored.json: the canonical dataset
# stores one iteration count per method and this figure is about the sweep. It
# prefers simulation_comparison_{physics}_10k and falls back to the original
# 750/1500 sweep, printing which it used.
fig_convergence.build(FIGS);        built("E5 convergence and compute")

fig_attention.build(FIGS);          built("E3 attention maps + bias control")
# E6 recovers the 28x28 arrays from the 2026-02-18 run's archived renders;
# see the module docstring for the resolution rationale.
fig_generalisation.build(FIGS); built("E6 generalisation to unseen patterns")
pending("E7 optical setup", "beam splitter orientation and AC254 typo fixed", "drawn asset")
# E8 loads ~220 DSLR frames; expect a few minutes.
fig_grid.build(FIGS);               built("E8 qualitative grid, PSNR annotated")
# E9 reads scored_aberration.json written by the Fig 5 cell above.
fig_aberration.build(FIGS);         built("E9 aberration severity sweep")

**The regime comparison, visually.** Companion to T4: for every canonical target, the same target's phase-only
hologram under Fresnel (z = 10 cm, rig modulator parameters) and under
Fraunhofer, plus the pi-patch locality probe. Near-field phase visibly
retains the scene; far-field phase is structureless, and a localized
modulator defect disturbs a spot in one regime and the whole reconstruction
in the other. Illustrative simulation; nothing here enters a quantitative
claim.


In [ ]:
import fig_regimes

# ~25 s per target (240 FFTs at 1000x1000), all 18 targets ~8 min.
# Presentation decisions and the pi-patch rationale live in the docstring.
fig_regimes.build(FIGS); built("Regime comparison, one per target")


## 7 - Build manifest

What this run produced, what is outstanding, and the provenance stamp that ties
every artifact to one scoring pass. Record the `sail_scored.json` timestamp
alongside any figure that leaves this notebook: if the dataset is re-scored,
figures built from the previous pass are stale and this is how that is caught.

In [ ]:
d = S.load()
print(f"scored dataset : {d['_path']}")
print(f"written        : {d['meta']['written']}")
print(f"records        : {len(d['records'])}\n")

print(f"BUILT ({len(BUILT)}):")
for b in BUILT:
    print(f"  {b}")
print(f"\nOUTSTANDING ({len(PENDING)}):")
for name, reason, blocker in PENDING:
    print(f"  {name:52s} {blocker or reason}")

bench_blocked = [n for n, _, b in PENDING if b == "bench session"]
if bench_blocked:
    print(f"\nOne bench session clears {len(bench_blocked)}: "
          f"{', '.join(bench_blocked)}.\n  Misalignment re-capture and the "
          f"aberration sweep share a rig configuration, so they are captured\n"
          f"  together rather than aligning the bench twice.")

manifest = {"scored": d["_path"], "written": d["meta"]["written"],
            "records": len(d["records"]), "built": BUILT,
            "pending": [{"item": n, "reason": r, "blocker": b}
                        for n, r, b in PENDING]}
# OUT_LIVE, not OUT. This manifest describes THIS run, so in verification
# mode it belongs beside the regenerated output rather than overwriting the
# deposited copy, whose paths were scrubbed at build time and whose contents
# record the run that produced the deposit.
(OUT_LIVE / "build_manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\nmanifest -> {OUT_LIVE / 'build_manifest.json'}")

## 8 | Deposit verification, regenerated against deposited

Only meaningful in verification mode. Compares every regenerated record
against the deposited `sail_scored.json` and `scored_aberration.json`.
GD-arm records are reported as expectedly missing when the deposit cites
them to the gdcitl record instead of carrying the captures.


In [ ]:
import json as _json

if not VERIFY_MODE:
    print('fresh-tree run, nothing to compare')
else:
    def _key(r):
        return (r['domain'], r['physics'], r['method'], r['target'])
    dep = {_key(r): r for r in _json.load(open(SCORED))['records']}
    reg = {_key(r): r for r in _json.load(open(OUT_LIVE / 'sail_scored.json'))['records']}
    shared = sorted(set(dep) & set(reg))
    worst = max((abs(dep[k]['psnr'] - reg[k]['psnr']) for k in shared), default=0.0)
    exact = sum(dep[k]['psnr'] == reg[k]['psnr'] for k in shared)
    print(f'sail_scored, deposited {len(dep)} records, regenerated {len(reg)}')
    print(f'  shared {len(shared)}, exact-psnr {exact}, worst |delta| {worst:.6f} dB')
    # A deposited record with no regenerated counterpart is only ever
    # legitimate for the GD-arm BENCH captures, and only when the deposit
    # cites them to the gdcitl record instead of carrying the files.
    # Anything else means this tree's code does not reproduce the record
    # this tree ships, which is the one thing this notebook exists to
    # catch, so it is an assertion rather than a printed caveat.
    CITED_GD_ARMS = {'gd', 'gd_citl_random', 'gd_citl_warm'}
    missing = sorted(set(dep) - set(reg))
    unexplained = sorted({f'{k[0]}/{k[2]}' for k in missing
                          if not (k[0].startswith('bench')
                                  and k[2] in CITED_GD_ARMS)})
    if missing:
        methods = sorted({k[2] for k in missing})
        print(f'  {len(missing)} deposited records not regenerated, methods {methods}.')
        if unexplained:
            print(f'  ** UNEXPLAINED: {unexplained} **')
            print('  These are not GD-arm bench captures, so the gdcitl citation')
            print('  does not account for them. The usual cause is this tree')
            print('  carrying analysis code older than the code that produced')
            print('  the deposited scoring pass.')
        else:
            print('  All are GD-arm bench captures cited to the gdcitl deposit')
            print('  rather than carried here, which is expected.')
    extra = sorted(set(reg) - set(dep))
    for k in extra[:10]:
        print('  EXTRA, investigate,', k)

    # Timestamps AND provenance are ignored: source_path/run_dir are
    # machine-local absolute paths in a regenerated record but scrubbed
    # relative paths in the deposited one, and meta['note'] names the scored
    # file by its deposit-era name. The numeric payload is compared exactly.
    def _strip(o):
        if isinstance(o, dict):
            return {k: _strip(v) for k, v in o.items()
                    if not any(w in k.lower() for w in
                               ('date', 'written', 'time', 'path', 'dir', 'note'))}
        if isinstance(o, list):
            return [_strip(v) for v in o]
        return o
    ab_dep, ab_reg = OUT / 'scored_aberration.json', OUT_LIVE / 'scored_aberration.json'
    if ab_dep.exists() and ab_reg.exists():
        same = (_json.dumps(_strip(_json.load(open(ab_dep))), sort_keys=True)
                == _json.dumps(_strip(_json.load(open(ab_reg))), sort_keys=True))
        print('scored_aberration identical (timestamps and provenance ignored),', same)
        assert same, 'the aberration record does not reproduce'
    assert not extra, 'regeneration produced records the deposit does not contain'
    assert not unexplained, (
        f'{unexplained} are in the deposited scoring pass but were not regenerated; '
        f'the deposited code does not reproduce the deposited record')
    assert worst < 0.005, 'regenerated PSNR diverges from the deposited record'
    print('VERIFICATION PASSED'
          + (' with cited GD-arm records skipped' if missing else ', every record reproduced'))
